# The Automatic Decomposition Advisor

Many optimization models have *structure* a solver can exploit: fixing a few
**complicating variables** disconnects the problem into independent recourse
blocks (**Benders decomposition** {cite:p}`Benders1962`, and its nonlinear
generalization **GBD** {cite:p}`Geoffrion1972`); a few **coupling constraints**
link otherwise-independent blocks (**Lagrangian relaxation**
{cite:p}`Geoffrion1974`, **Dantzig–Wolfe** {cite:p}`DantzigWolfe1960`).

Historically the modeler had to *recognize* the structure and *rewrite* the model
by hand. The **Decomposition Advisor** automates that: it analyzes a model,
discovers exploitable structure as a graph problem, scores candidate
decompositions, explains its recommendation, and produces a solvable reformulation
— a compiler-style transformation pass on the same footing as presolve. See the
design spec in `docs/design/decomposition-advisor.md`; for a broader survey of
Benders methods see {cite:t}`Rahmaniani2017`.

This notebook walks the advisor's analysis surface end to end. It is pure
analysis — no solve is triggered — so it runs without the compiled solver
backend; running the chosen decomposition is covered in the
[Benders](tutorial_benders) and [Lagrangian](tutorial_lagrangian) tutorials.

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import discopt.modeling as dm
from discopt.decomposition import RecordStore, analyze_decomposition, record_outcome
from discopt.decomposition.learning import ObservedPerformance, Outcome

## A two-stage facility model

Three candidate facilities: binary `build[i]` decides whether to open facility
`i` (first stage), continuous `x[i]` is its production (recourse). Production is
gated by the build decision, and a budget limits how many we open — the budget
row **couples the binaries**, so the model is *not* already block-diagonal. But
once the binaries are fixed, the recourse splits into three independent blocks:
the textbook Benders structure.

In [ ]:
m = dm.Model("facility")
# Scalar variables so each facility is a distinct node in the structure graph.
b = [m.binary(f"b{i}") for i in range(3)]                    # first-stage: open i?
x = [m.continuous(f"x{i}", lb=0, ub=10) for i in range(3)]   # recourse: production

for i in range(3):
    m.subject_to(x[i] <= 10 * b[i])                          # production gated by build

m.subject_to(b[0] + b[1] + b[2] <= 2)                        # budget couples the binaries
m.minimize((x[0] + x[1] + x[2]) - 3 * (b[0] + b[1] + b[2]))
print(m)

## Analyze the structure

`model.analyze_decomposition()` returns a `DecompositionAdvisor`. Its
`StructureReport` summarizes the exploitable structure in one pass — sizes,
integer localization (how many recourse blocks appear once the integers are
fixed), coupling density, and more.

In [ ]:
adv = m.analyze_decomposition()
print(adv.structure().summary())

## Candidate decompositions

The advisor discovers candidate decompositions with a cheap-first cascade of
generators. Each candidate reuses the existing `DecompositionStructure`, so it
can drive the shipping solvers directly. The monolithic *no-decomposition*
baseline is always included.

In [ ]:
for c in adv.candidates():
    print("-", c.summary())

## Score and rank

Scoring is method-aware and measured against the no-decomposition baseline
(anchored at 0): a decomposition must *beat* solving the model as written. The
score carries an analytic performance estimate (an Amdahl-style speedup capped by
load balance) and a confidence. Unsound candidates are vetoed, not down-weighted.

In [ ]:
for cand, sv in adv.scores():
    print(f"{cand.method.label:22s} {sv.summary()}")

## The recommendation, explained

Every recommendation is explainable — the reasons are backed by the same metrics
the scorer used, with concerns and the runner-up alternatives. The explanation
renders as text, markdown, or JSON (the JSON feeds tooling / the LLM layer).

In [ ]:
print(adv.explain())

In [ ]:
print(adv.explain("json"))

### Counterfactuals: "why *not* method X?"

The advisor also explains why a method was **not** chosen — including the missing
structural precondition. Here the budget row couples the binaries, so the model is
*not* already block-diagonal and an independent-block decomposition does not apply.
(Lagrangian relaxation, by contrast, *is* a viable alternative — it appears as a
runner-up above, since the budget row can be dualized.)

In [ ]:
print(adv.explain(method="independent_blocks"))

## Reformulate: build a solvable decomposition

`decompose()` turns the recommended candidate into a `DecomposedModel` — a
master/subproblem partition with a **soundness certificate** and an
original-space **variable mapping**. Its `solve()` (not called here) dispatches to
the shipping Benders / GBD / Lagrangian drivers.

In [ ]:
dcmp = adv.decompose()
print(dcmp.summary())
print()
print("variable roles:")
for name in ["b0", "b1", "b2", "x0", "x1", "x2"]:
    print(f"  {name}: {dcmp.var_map.role(name)}")

## Parallel execution plan

The independent recourse blocks run on a pluggable communication backend
(sequential or thread pool today), ordered biggest-first to avoid stragglers. The
`SchedulingGraph` reports the load-balance speedup cap; `map_subproblems` runs a
per-block function and reduces results **back into block order**, so the answer is
deterministic regardless of backend. Here we map a trivial pure-Python function
(the block size) rather than a real solve.

In [ ]:
print(dcmp.schedule().summary())
block_sizes = dcmp.map_subproblems(lambda sp: sp.size, backend="threads")
print("recourse block sizes (block order):", block_sizes)

## Learn from experience

Every solve can be recorded — the instance's features, the chosen method, what
was *predicted*, and what was *observed* — so the advisor's own error is
measurable and future recommendations can improve. Records persist as local JSON
lines; an `InstanceBasedPolicy` retrieves nearest past instances to bias
selection (deferring safely to the rule-based policy until enough data accrues).

In [ ]:
store = RecordStore()  # in-memory; pass a path to persist as JSONL
record_outcome(
    adv,
    store,
    observed=ObservedPerformance(wall_clock_s=1.8, iterations=6),
    outcome=Outcome.OPTIMAL,
    timestamp=0.0,
)
rec = store.all()[0]
print("chosen           :", rec.chosen)
print("predicted speedup:", rec.predicted_speedup)
print("features         :", rec.features.to_dict())

## Summary

The advisor took a model and, without any manual rewriting, discovered its
two-stage structure, scored the candidates against the no-decomposition baseline,
recommended **Benders** with an evidence-backed rationale, and produced a
solvable `DecomposedModel` with a soundness certificate and a parallel execution
plan — then recorded the outcome for future learning.

Future directions (design spec §14) include automatic hypergraph partitioning for
block-angular models {cite:p}`KarypisKumar1998`, learned decomposition policies,
and graph-neural-network structure prediction.